# Code to Import and to Manage Input Files to the Cost-Benefit Analysis of the air pollution scenarios supporting the revision of Air Convention Gothenburg Protocol

**Python version:** Python 3.11 or newer (tested with Python 3.11.15)
**Compiler:** JupyterLab or classic Jupyter Notebook  

**Dependencies:** see [`requirements.txt`](../requirements.txt) in the repository root (install with `pip install -r requirements.txt`).

# Packages

In [9]:
# === PART 1: PACKAGE IMPORTS ===
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from scipy.interpolate import interp1d

# Tabular list

Below is a table containing a description and source overview of the input files for the CBA, available in the Raw Data folder

| CBA Section | Name | File/Source |Download date|
|:--------|:--------|:--------|:--------|
|  General   |  Regional Allocation   |  12 <br> https://tool.valesor.eu/   | 31st of August 2026 |
|  General   |  Population in 5 yr Cohorts   |  8 <br> https://tool.valesor.eu/   |31st of August 2026|
|  General   |  Life Expectancy at 5 year Cohorts   |  WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_FULL <br> UN World population prospects   |31st of August 2026|
|  General   |  Air Pollution Costs   |  LRTAP_ctry_GNFR_APcost_v7a_090926_0 <br> TFIAM Web page   |31st of August 2026|
|  Biodiversity   |  Exceedence of CL in Forest and Seminatural ecosystems   |  v7a_2040_clempn_exc_allecosys <br> GAINS Explorer online   |31st of August 2026|
|  Fertilisation   |  Non-exceedence of CL in Forest ecosystems   |  v7a_2040_clempn_exc_allecosys <br> GAINS Explorer online |31st of August 2026|
|  Climate   |  Emission levels   |  LRTAP_ctry_GNFR_emis_v7a_090926_0 <br> TFIAM Web page   |31st of August 2026|
|  Climate   |  Emissions Damage Costs   |  SLCF_econ_value <br> https://doi.org/10.1017/bca.2023.8   |31st of August 2026|
|  Climate   |  Emissions Climate Forcing Values   |  Climate_Metric <br> https://doi.org/10.1017/bca.2023.8   |31st of August 2026|
|  Health   |  PM2.5 Concentrations   |  v7a_2040_yoll_Europe <br> GAINS Explorer online   |31st of August 2026|
|  Health   |  Exposure Respone Function (ERF)   |  Health_ERF <br> https://tool.valesor.eu/   |31st of August 2026|
|  Health   |  Morbidity Costs   | 12/Nordic_mor_cost <br> https://tool.valesor.eu/ & https://doi.org/10.1787/76ca89a2-en   |31st of August 2026|
|  Health   |  Mortality Costs   |  12/Nordic_mor_cost <br> https://tool.valesor.eu/ & https://doi.org/10.1787/76ca89a2-en   |31st of August 2026|
|  Health   |  Mortality Incidents Rate per Cohort   |  8 <br> https://tool.valesor.eu/ & https://doi.org/10.1787/76ca89a2   |31st of August 2026|
|  Health   |  Morbidity Incidents Rate per Cohort   |  8 <br> https://tool.valesor.eu/ & https://doi.org/10.1787/76ca89a2   |31st of August 2026|

# Import files

In this section, all imports of raw data files presented, following the CBA section the import belongs to. 


In [10]:
# data version
DATA_VERSION = "v7a"

**Scenario-specific** files which are imported depending on the chosen scenario

In [11]:
FILES = {
    "v7a": {
        "AP": r"Raw_Data/LRTAP_ctry_GNFR_APcost_v7a_250928.xlsx",
        "emissions": r"Raw_Data/LRTAP_ctry_GNFR_emis_v7a_250928.xlsx",
        "health_2040": r"Raw_Data/v7aa_2040_yoll_Europe.xlsx",
        "health_2050": r"Raw_Data/v7aa_2050_yoll_Europe.xlsx",
        "biodiv_2040": r"Raw_Data/v7aa_2040_clempn_exc_allecosys.xlsx",
        "biodiv_2050": r"Raw_Data/v7aa_2050_clempn_exc_allecosys.xlsx",
        "EMEP_data_v7": r"Raw_Data/EMEP_country_PM10_POD1_POD3_v7.xlsx",
        "EMEP_data_v5": r"Raw_Data/EMEP_country_PM10_POD1_POD3_v5.xlsx"
    }
}

## General

In [12]:
# Data common for all scenarios

# Country-region allocation from VALESOR
# FIND FILE via the VALESOR online tool support files https://tool.valesor.eu/ 
df_countries = pd.read_excel(r"Raw_Data/12.xlsx", sheet_name='Countries')

# Load Population data from VALESOR
# FIND FILE via the VALESOR online tool support files https://tool.valesor.eu/ 
df_pop_BL = pd.read_excel(r"Raw_Data/8.xlsx", sheet_name='Population',  skiprows=[0])

# Load Population data from UN World Population Prospects (truncated version)
# FIND FILE via UN World Population Prospects 2024 https://population.un.org/wpp/
LE_file_name = r"Raw_Data/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_FULL.xlsx"

# Load AP Control costs per country, year, scenario, GNFR sector as well as emissions: 
# FIND FILE at TFIAM Web page https://iiasa.ac.at/policy/applications/task-force-on-integrated-assessment-modelling-tfiam-under-lrtap-convention
df_AP = pd.ExcelFile(FILES[DATA_VERSION]["AP"])

## Biodiversity

In [13]:
# Load Excel file for BIO_CL-concentrations
wb_biodiv_2040 = load_workbook(FILES[DATA_VERSION]["biodiv_2040"], data_only=True)
wb_biodiv_2050 = load_workbook(FILES[DATA_VERSION]["biodiv_2050"], data_only=True)

# Import damage cost file
df_biodiv_costs = pd.read_excel(r"Raw_Data/New_biodiv_damage_costs.xlsx")

## Climate

In [14]:
# IMPORT SLCF ECONONOMIC CLIMATE CHANGE DAMAGE COSTS
df_em_costs = pd.read_excel(r"Raw_Data/SLCF_econ_value.xlsx", sheet_name="C-cost per pollutant")

# IMPORT SLCF CLIMATE METRICS
df_climate_metrics = pd.read_excel(r"Raw_Data/Climate_Metric.xlsx")

# Load Excel file for emissions
emxls = pd.ExcelFile(FILES[DATA_VERSION]["emissions"])

## Health

Load PM2.5 annual average mean population weighted concentration per country, year, and scenario: 
FIND raw data Scenario-specific files by exporting PM2.5 health results as .csv-files for each scenario from 
https://gains.iiasa.ac.at/gains/GPV/index.login?logout=1
then open each .csv-file. After this, 'move' each .csv to a common xlsx.file. 
name the excel file "PM_conc_exp" and save in the 'Raw_Data' folder

In [15]:
# IMPORT Health ERF
df_H_ERF = pd.read_excel(r"Raw_Data/Health_ERF.xlsx", sheet_name='AIR')

# IMPORT Health morbidity
df_morb_costs = pd.read_excel(r"Raw_Data/12.xlsx", sheet_name='VALESOR Morbidity')
df_morb_nordics_costs = pd.read_excel(r"Raw_Data/NORDIC_mor_cost.xlsx", sheet_name='Morb')

# IMPORT Health mortality
df_mort_costs_oecd = pd.read_excel(r"Raw_Data/Valesor 12_updated with OECD values.xlsx", sheet_name='VALESOR VOLY VSL vQALY')

# IMPORT health effect data
df_morb_eff = pd.read_excel(r"Raw_Data/8.xlsx", sheet_name='Morbidity',  skiprows=[0])
df_mort_spec_eff = pd.read_excel(r"Raw_Data/8.xlsx", sheet_name='Cause-specific mortality',  skiprows=[0])
df_mort_all_eff = pd.read_excel(r"Raw_Data/8.xlsx", sheet_name='All-cause mortality',  skiprows=[0])

# Load Excel file for population-weighted PM25-concentrations
wb_pm_conc_2040 = load_workbook(FILES[DATA_VERSION]["health_2040"], data_only=True)
wb_pm_conc_2050 = load_workbook(FILES[DATA_VERSION]["health_2050"], data_only=True)

## Crops indicator based on O3 conc in v5 ex post analysis by WGE & v7 scenarios

In [16]:
# Load Excel file for production losses of crops as function of O3 concentration (indicated by eutrophying N dep)
avoided_prod_losses = pd.read_excel(r"Raw_Data/Wheat_Potato_av_losses.xlsx", sheet_name="To determine")
Extr_values = pd.read_excel(r"Raw_Data/Wheat_Potato_av_losses.xlsx", sheet_name="Extrapolation values")
EMEP_POD3_v5 = pd.read_excel(FILES[DATA_VERSION]["EMEP_data_v5"]).drop(columns=["PM10_surface_ug_m3", "POD1", "PM10_variable", "POD1_variable"])

# Load part of Excel file for ground level ozone
EMEP_POD3_v7 = pd.read_excel(FILES[DATA_VERSION]["EMEP_data_v7"]).drop(columns=["PM10_surface_ug_m3", "POD1", "PM10_variable", "POD1_variable"])

## Carbon sequestration indicator based on O3 conc in v5 ex post analysis by WGE & v7 scenarios

In [17]:
# Load Excel file for production losses of crops as function of O3 concentration (indicated by eutrophying N dep)
avoided_carbon_losses = pd.read_excel(r"Raw_Data/carbon_losses_values.xlsx", sheet_name="To determine")
Extr_carbon_values = pd.read_excel(r"Raw_Data/carbon_losses_values.xlsx", sheet_name="Extrapolation values")
EMEP_POD1_v5 = pd.read_excel(FILES[DATA_VERSION]["EMEP_data_v5"]).drop(columns=["PM10_surface_ug_m3", "POD3", "PM10_variable", "POD3_variable"])

# Load part of Excel file for ground level ozone
EMEP_POD1_v7 = pd.read_excel(FILES[DATA_VERSION]["EMEP_data_v7"]).drop(columns=["PM10_surface_ug_m3", "POD3", "PM10_variable", "POD3_variable"])

## Soiling indicator

In [18]:
# Load files for surface areas per citizens
Soiling_area_ICP = pd.read_excel("Raw_Data/Surface_area_per_country_citizen.xlsx", sheet_name="S_area")
Soiling_area_Project_est = pd.read_excel("Raw_Data/Additional surface area data.xlsx")

# Load part of Excel file for PM10
EMEP_PM10 = pd.read_excel(FILES[DATA_VERSION]["EMEP_data_v7"]).drop(columns=["POD1", "POD3", "POD1_variable", "POD3_variable"])

# Table Conversion - General

Below follows all general data management supporting the subsequent CBA calculations

In [19]:
# =====================================================
# Scenario Descriptions
# =====================================================

MAPPING_SCENARIOS = {
    "Baseline_v7a": "Base_v7a",
    "OPT Health -50 AAE -35 v7a": "Health 50%, Bio 35%-v7a",
    "OPT Health -50 AAE -40 v7a": "Health 50%, Bio 40%-v7a",
    "OPT Health -50 AAE -45 v7a": "Health 50%, Bio 45%-v7a",
    "OPT Health -50 AAE -50 v7a": "Health 50%, Bio 50%-v7a",
    "OPT Health -55 AAE -40 v7a": "Health 55%, Bio 40%-v7a",
    "OPT Health -55 AAE -45 v7a": "Health 55%, Bio 45%-v7a",
    "OPT Health -55 AAE -50 v7a": "Health 55%, Bio 50%-v7a",
    "OPT Health -60 AAE -40 v7a": "Health 60%, Bio 40%-v7a",
    "OPT Health -60 AAE -45 v7a": "Health 60%, Bio 45%-v7a",
    "OPT Health -60 AAE -50 v7a": "Health 60%, Bio 50%-v7a",
    "J5035/d/D/v7a": "Health 50%, Bio 35%-v7a",
    "J5040/d/D/v7a": "Health 50%, Bio 40%-v7a",
    "J5045/d/D/v7a": "Health 50%, Bio 45%-v7a",
    "J5050/d/D/v7a": "Health 50%, Bio 50%-v7a",
    "J5540/d/D/v7a": "Health 55%, Bio 40%-v7a",
    "J5545/d/D/v7a": "Health 55%, Bio 45%-v7a",
    "J5550/d/D/v7a": "Health 55%, Bio 50%-v7a",
    "J6040/d/D/v7a": "Health 60%, Bio 40%-v7a",
    "J6045/d/D/v7a": "Health 60%, Bio 45%-v7a",
    "J6050/d/D/v7a": "Health 60%, Bio 50%-v7a",
    "J5035/s/D/v7a": "Health 50%, Bio 35%-v7a",
    "J5040/s/D/v7a": "Health 50%, Bio 40%-v7a",
    "J5045/s/D/v7a": "Health 50%, Bio 45%-v7a",
    "J5050/s/D/v7a": "Health 50%, Bio 50%-v7a",
    "J5540/s/D/v7a": "Health 55%, Bio 40%-v7a",
    "J5545/s/D/v7a": "Health 55%, Bio 45%-v7a",
    "J5550/s/D/v7a": "Health 55%, Bio 50%-v7a",
    "J6040/s/D/v7a": "Health 60%, Bio 40%-v7a",
    "J6045/s/D/v7a": "Health 60%, Bio 45%-v7a",
    "J6050/s/D/v7a": "Health 60%, Bio 50%-v7a",
    "MTFR_v7a": "MTFR_v7a"    
}

## Regional Allocation

Creating a country-region mapping to be used for grouping benefits by either country-specific values or for entire regions. Region allocation is based on TFIAM/CIAM policy briefs and contains the following groups: EU27, EFTA (Iceland, Norway, Switzerland, United Kingdom), Türkiye, EECCA (Armenia, Azerbaijan, Kazakhstan, Kyrgyzstan, Uzbekistan, Turkmenistan, Tajikistan, Georgia, Ukraine, Belarus, Russia, Republic of Moldova), West-Balkan (Albania, Bosnia and Herzegovina, Kosovo, Republic of North Macedonia, Montenegro, Serbia).

In [20]:
# =====================================================
# Region definitions
# =====================================================

EU27 = [
    "Austria","Belgium","Bulgaria","Croatia","Cyprus",
    "Czech Rep.","Denmark","Estonia","Finland","France",
    "Germany","Greece","Hungary","Ireland","Italy","Latvia",
    "Lithuania","Luxembourg","Malta","Netherlands","Poland",
    "Portugal","Romania","Slovakia","Slovenia",
    "Spain","Sweden"
]

EFTA = [
    "Iceland","Norway","Switzerland","United Kingdom"
]

TURKIYE = [
    "Türkiye"
]

EECCA = [
    "Armenia","Azerbaijan","Kazakhstan","Kyrgyzstan",
    "Uzbekistan","Turkmenistan","Tajikistan","Georgia",
    "Ukraine","Belarus","Russia","R Moldova"
]

WEST_BALKAN = [
    "Albania",
    "Bosnia-H",
    "Kosovo",
    "North Macedonia",
    "Montenegro",
    "Serbia"
]

# =====================================================
# Region nd Country Mappings
# =====================================================

REGION_COLS = {
    "EU27": "EU27",
    "EECCA": "EECCA",
    "EFTA+UK": "EFTA",
    "UNECE Europe": "UNECE Europe", 
    "UNECE Caucasus + Asia": "UNECE Caucasus + Asia",
    "West Balkan": "Western Balkan"
}

MAPPING_COUNTRIES = {"Bosnia and Herzegovina": "Bosnia-H",
            "Czechia": "Czech Rep.",
            "Kosovo (under UNSC res. 1244)": "Kosovo",
            "Republic of Moldova": "R Moldova",
            "Moldova": "R Moldova",
            "Republic of North Macedonia": "North Macedonia",
            "Russian Federation": "Russia",
            "Turkey": "Türkiye",
            "Slovak Republic": "Slovakia",
            "Czech Republic": "Czech Rep.",
            "Russia European part": "Russia",
            "Pan-European": "UNECE (Excluding North America)",
            "Western Balkan": "West Balkan",
            "EFTA": "EFTA+UK"}

In [21]:
# --- Build region -> countries dictionary from the allocation table ---
Regions = {}

for region, col in REGION_COLS.items():
    Regions[region] = df_countries.loc[
        df_countries[col].notna(), "Country"
    ].tolist()

# --- Create additional regions or adjust existing ones ---

# Countries not belonging to EU27 (+ Kosovo manually added)
Regions["NON EU27"] = (
    df_countries.loc[
        df_countries["EU27"].isna(), "Country"
    ].tolist()
    + ["Kosovo"]
)

# Add Kosovo to existing regions
Regions["UNECE Europe"].append("Kosovo")
Regions["West Balkan"].append("Kosovo")

Regions["EFTA+UK"].append("United Kingdom")

# Manually defined regional groups
Regions["Nordics"] = ["Sweden", "Denmark", "Norway", "Finland", "Iceland"]

# --- Convert dictionary to dataframe (long format: Region–Country pairs) ---
region_df = pd.DataFrame(
    [(region, country) for region, countries in Regions.items() for country in countries],
    columns=["Region", "Country"]
)

# Identify unique countries
pan_eu = region_df[['Country']].drop_duplicates().copy()

# Add new region
pan_eu['Region'] = 'UNECE (Excluding North America)'

# Merge with existing region dataframe
region_df = pd.concat([region_df, pan_eu], ignore_index=True)

region_df = region_df.replace(MAPPING_COUNTRIES)

In [22]:
# Save the df as an Excel file
region_df.to_excel(r"Processed_Data/region_allocation_check.xlsx", index=False)

## Population in 5 Year Cohorts

Recalculate the population values in 5 year cohorts to match format of other age dependent data.

In [23]:
# Select columns that are integers (age columns)
age_cols_pop = [c for c in df_pop_BL.columns if isinstance(c, int)]

# Create age groups: 0, 1, 2-4, then 5-year intervals
groups = {
    0: [0],
    1: [1],
    "2-4": [2, 3, 4]
}

# Continue with normal 5-year groups from age 5
groups.update({
    f"{i}-{i+4}": list(range(i, i+5))
    for i in range(5, 100, 5)
})

# Sum columns within each group
df_5yr = pd.concat(
    [df_pop_BL.drop(columns=age_cols_pop),
     pd.DataFrame({name: df_pop_BL[cols].sum(axis=1) for name, cols in groups.items()})],
    axis=1
)

df_5yr.rename(columns={"95-99": "95+"}, inplace=True)

df_5yr = df_5yr.drop(columns=["Life expectancy"])

df_5yr = df_5yr.replace(MAPPING_COUNTRIES)

# Melt the df as to only have one value column
df_5yr_melt = df_5yr.melt(
    id_vars=["Year", "Country"],
    var_name="5yr cohort",
    value_name="pop_value"
)

df_5yr_melt = df_5yr_melt.replace(MAPPING_COUNTRIES)

In [24]:
# Save the df as an Excel file
df_5yr_melt.to_excel(r"Processed_Data/POP_5yr.xlsx", index=False)

## Life Expectancy at 5 year Cohorts

The code below uses UN values for life expectancy at four ages (birth, 15, 65 and 80) to interpolate the life expectancy at all ages between 0 and 100 and then grouping them by 5 year cohorts. The interpolation is done via a third degree polynomial and if a LE value is negative, it is replaced by 1.

### LE Input Cleaning

In [25]:
# Columns to read from the Excel file
use_cols = [
    "Variant",
    "Region, subregion, country or area *",
    "Type",
    "Year",
    "Life Expectancy at Birth, both sexes (years)",
    "Life Expectancy at Age 15, both sexes (years)",
    "Life Expectancy at Age 65, both sexes (years)",
    "Life Expectancy at Age 80, both sexes (years)",
]

# Sheets to be read from the Excel file
selected_sheets = ["Estimates", "Medium variant"]

all_dfs = []
for sh in selected_sheets:
    # Read selected columns from each sheet, using row 16 as header
    df = pd.read_excel(
        LE_file_name,
        sheet_name=sh,
        header=16,
        usecols=use_cols,
        engine="openpyxl"
    )

    # Replace "Medium" with "Mid" in the Variant column
    df = df.replace({"Variant": {"Medium": "Mid"}})

    # Rename columns for consistency
    df = df.rename(columns={
        "Variant": "Life_exp_level",
        "Region, subregion, country or area *": "Country"
    })

    # Filter for countries and selected years only
    df = df[(df["Type"] == "Country/Area") & (df["Year"].isin([2020, 2030, 2040, 2050]))].copy()

    # Append processed DataFrame to list
    all_dfs.append(df)

# Concatenate all DataFrames into one
input_LE_values = pd.concat(all_dfs, ignore_index=True)

# Replace country names based on mapping
input_LE_values = input_LE_values.replace(MAPPING_COUNTRIES)

### Interpolation

In [26]:
# -----------------------------
# Use a third-degree polynomial equation to intrapolate the life expactancy at year 30
# -----------------------------

# Define age range from 0 to 99
ages = np.arange(0, 100)

def estimate_at_age(row):
    # Convert row values to numeric and ensure float type
    y = pd.to_numeric(row, errors="coerce").values.astype(float)
    
    # Fit a 3rd-degree polynomial using known life expectancy points
    coeffs = np.polyfit([0, 15, 65, 80], y, 3)
    poly = np.poly1d(coeffs)
    
    # Evaluate polynomial for all ages and return as a Series
    return pd.Series(poly(ages), index=ages)

# Columns containing life expectancy values at specific ages
life_cols = [
    "Life Expectancy at Birth, both sexes (years)",
    "Life Expectancy at Age 15, both sexes (years)",
    "Life Expectancy at Age 65, both sexes (years)",
    "Life Expectancy at Age 80, both sexes (years)"
]

# Apply interpolation function row-wise
LE_at_age = input_LE_values[life_cols].apply(estimate_at_age, axis=1)

# Rename columns to indicate life expectancy per age
LE_at_age.columns = [f"LE_{age}" for age in ages]

# Concatenate interpolated values with original DataFrame
LE_middleman = pd.concat([input_LE_values, LE_at_age], axis=1)

# Drop original life expectancy columns and unnecessary columns
out_LE_at_each_age = LE_middleman.drop(columns=[
    "Type",
    "Life Expectancy at Birth, both sexes (years)",
    "Life Expectancy at Age 15, both sexes (years)",
    "Life Expectancy at Age 65, both sexes (years)",
    "Life Expectancy at Age 80, both sexes (years)"
])

# If the age value is <1, gradually decay it as 80% of the previous value
age_cols_LE = [f"LE_{age}" for age in range(100)]
decay_rate = 0.8

for idx in out_LE_at_each_age.index:
    row = out_LE_at_each_age.loc[idx, age_cols_LE].copy()
    
    for age in range(0, 100):
        col = f"LE_{age}"
        prev_col = f"LE_{age-1}"
        
        if pd.isna(row[col]) or row[col] <= 1:
            # From this age onward, apply 80% decay
            for next_age in range(age, 100):
                row[f"LE_{next_age}"] = decay_rate * row[f"LE_{next_age-1}"]
            break
    
    out_LE_at_each_age.loc[idx, age_cols_LE] = row

### Grouping by Age Cohorts

In [27]:
# Merge region information into the life expectancy DataFrame
tmp = out_LE_at_each_age.merge(
    region_df,
    on="Country",
    how="left"
)

# Remove unnecessary population columns
df_pop_short = df_pop_BL.drop(columns=["Life expectancy", "Total"])

# Replace country names based on mapping
df_pop_short = df_pop_short.replace(MAPPING_COUNTRIES)

# Merge population data with life expectancy data
KEYS = ["Country", "Year"]

out_life_exp_weighted = tmp.merge(df_pop_short, on=KEYS, how="inner")

# Multiply life expectancy by population for each age
for a in ages:
    out_life_exp_weighted[a] = out_life_exp_weighted[f"LE_{a}"] * out_life_exp_weighted[a]

# Select columns with integer names (single-year age columns)
age_cols_int = [c for c in out_life_exp_weighted.columns if isinstance(c, int)]

# Define age groups: 0, 1, 2-4, and then 5-year intervals
groups = {
    0: [0],
    1: [1],
    "2-4": [2, 3, 4]
}

# Add 5-year age groups from age 5 to 99
groups.update({
    f"{i}-{i+4}": list(range(i, i+5))
    for i in range(5, 100, 5)
})

# Sum weighted values within each age group
summed_weighted_LE = pd.concat(
    [out_life_exp_weighted.drop(columns=age_cols_int),
     pd.DataFrame({name: out_life_exp_weighted[cols].sum(axis=1) for name, cols in groups.items()})],
    axis=1
)

# Rename the last age group from 95-99 to 95+
summed_weighted_LE.rename(columns={"95-99": "95+"}, inplace=True)

# Keep only identifier columns and aggregated age group columns
summed_weighted_LE_clean = pd.concat([summed_weighted_LE.iloc[:, :3], summed_weighted_LE.iloc[:, 103:]], axis=1)

# Merge grouped life expectancy values with grouped population data
df = summed_weighted_LE_clean.merge(df_5yr, on=["Country", "Year"], suffixes=("_LE", "_POP"))

# Identify age group columns
age_cols = [c for c in summed_weighted_LE_clean.columns if c not in ["Country", "Year", "Life_exp_level", "Region"]]

# Create result DataFrame with identifier columns
result = df[["Country", "Year", "Life_exp_level", "Region"]].copy()

# Divide weighted life expectancy by population to get average life expectancy per age group
for c in age_cols:
    result[c] = df[f"{c}_LE"] / df[f"{c}_POP"]

# Reshape the DataFrame to long format with one value column
result_melt = result.melt(
    id_vars=["Country", "Year", "Life_exp_level", "Region"],
    var_name="5yr cohort",
    value_name="LE_value"
)

order = list(groups.keys())
order[-1] = "95+" 

result_melt["5yr cohort"] = pd.Categorical(
    result_melt["5yr cohort"],
    categories=order,
    ordered=True
)

# Sort the final DataFrame by country
result_melt = result_melt.sort_values(["Country", "5yr cohort"])

In [28]:
result_melt

,Country,Year,Life_exp_level,Region,5yr cohort,LE_value
110,Albania,2020.0,Estimates,UNECE Europe,0,77.824000
111,Albania,2020.0,Estimates,West Balkan,0,77.824000
112,Albania,2020.0,Estimates,NON EU27,0,77.824000
113,Albania,2020.0,Estimates,UNECE (Excluding North America),0,77.824000
503,Albania,2030.0,Mid,UNECE Europe,0,80.839000
...,...,...,...,...,...,...
14760,Uzbekistan,2040.0,Mid,UNECE (Excluding North America),95+,1.944364
14761,Uzbekistan,2050.0,Mid,EECCA,95+,1.700867
14762,Uzbekistan,2050.0,Mid,UNECE Caucasus + Asia,95+,1.700867
14763,Uzbekistan,2050.0,Mid,NON EU27,95+,1.700867


In [29]:
# Save the df as an Excel file
result_melt.to_excel("Processed_Data/LE_5yr.xlsx", index=False)

## Air Pollution Abatement Costs

This section manages and cleans the data related to air pollution abatement costs. These are the values of air pollution damage costs are to be compared to in order to see if the benefits (reduced air pollution damage costs) outweight the abatement costs.

In [30]:
# Ignore first sheet
sheet_names = df_AP.sheet_names[1:]
scenario_version = pd.read_excel(df_AP, sheet_name="info", header=None)
rows = []

# =====================================================
# Process each country sheet
# =====================================================
for sheet in sheet_names:
    # Read sheet without headers (fixed cell positions)
    df = pd.read_excel(
        df_AP,
        sheet_name=sheet,
        header=None
    )

    # -----------------------------
    # Country (B1)
    # -----------------------------
    country = df.iloc[0, 1]

    # -----------------------------
    # Sector (B5:B18)
    # -----------------------------
    sectors = df.iloc[4:18, 1].tolist()
    
    # Determine the scenario version and year in use and use the correct rows/columns.
    
    if scenario_version.iloc[4, 1] == "Base_v7a":
        
        # -----------------------------
        # Scenario (C3:X3)
        # -----------------------------
        scenarios = df.iloc[2, 2:24].tolist()
       
        # -----------------------------
        # Year (C4:X4)
        # -----------------------------
        years = df.iloc[3, 2:24].tolist()

        # -----------------------------
        # Net AP Costs (C5:X18)
        # -----------------------------
        costs = df.iloc[4:18, 2:24].values
    
    # -----------------------------
    # Build stacked list
    # -----------------------------
    for i, sector in enumerate(sectors):
        for j, (scenario, year) in enumerate(zip(scenarios, years)):
            rows.append({
                "Country": country,
                "Year": year,
                "Scenario": scenario,
                "Sector": sector,
                "Net AP Costs": costs[i, j]
            })

# -----------------------------
# Turn the stacked list into a df
# -----------------------------
AP_c_df = pd.DataFrame(rows)

# =====================================================
# Add baseline scenarios
# =====================================================

# 1) Build the full set of keys we want Base to cover
base_keys = AP_c_df[["Country", "Year", "Sector"]].drop_duplicates()

if scenario_version.iloc[4, 1] == "Base_v7a":

    base_df = base_keys.assign(
        Scenario="Base_v7a",
        Net_AP_Costs= 0
    )

# 3) Append
AP_c_df = pd.concat([AP_c_df, base_df], ignore_index=True)

# Drop incomplete rows if any
AP_c_df = AP_c_df.dropna(
    subset=["Country", "Sector", "Year", "Scenario"]
).reset_index(drop=True)

# Convert units from million Euro to Euro
AP_c_df["Net AP Costs"] = AP_c_df["Net AP Costs"]*1000000

# Convert to common country & scenario names
AP_c_df = AP_c_df.replace(MAPPING_SCENARIOS)
AP_c_df = AP_c_df.replace(MAPPING_COUNTRIES)

In [31]:
# Save the df as an Excel file
AP_c_df.to_excel("Processed_Data/APcosts_check.xlsx", index=False)

# Table Conversion - Crops

Below follows all data management specifically pertaining to the Crop valuation parts of the CBA

In [32]:
# Filter out the rows which ends with the relevant scenario version in use

crops_out = avoided_prod_losses[
    avoided_prod_losses["Scenario"].str.endswith(DATA_VERSION.lower())
]
crops_out = crops_out.replace(MAPPING_SCENARIOS)

C:\Users\StefanÅström\AppData\Local\Temp\ipykernel_16676\1985525649.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  crops_out = crops_out.replace(MAPPING_SCENARIOS)


In [33]:
# Merge EMEP values with region_df
EMEP_POD3_v7_expanded = EMEP_POD3_v7.merge(region_df, on="Country", how="inner")
EMEP_POD3_v5_expanded = EMEP_POD3_v5.merge(region_df, on="Country", how="inner")

# Calculate the mean ozone (POD3) levels for each scenario in UNECE (Excluding North America) 
EMEP_POD3_v7_expanded = EMEP_POD3_v7_expanded[EMEP_POD3_v7_expanded["Region"]=="UNECE (Excluding North America)"]
EMEP_POD3_v7_UNECE_mean = EMEP_POD3_v7_expanded.groupby(["Region", "Scenario", "Year"], as_index=False)["POD3"].mean()

EMEP_POD3_v5_expanded = EMEP_POD3_v5_expanded[EMEP_POD3_v5_expanded["Region"]=="UNECE (Excluding North America)"]
EMEP_POD3_v5_UNECE_mean = EMEP_POD3_v5_expanded.groupby(["Region", "Scenario"], as_index=False)["POD3"].mean()
EMEP_POD3_v5_UNECE_mean = EMEP_POD3_v5_UNECE_mean.rename(columns={"Scenario": "Scenario_UKCEH"})

# Merge v7 to crops extrapolation of avioded losses
Crops_expanded = crops_out.merge(EMEP_POD3_v7_UNECE_mean, on=["Scenario", "Year"], how="inner")

# Merge v5 to extrapolation values
Crops_extrapolation_values = Extr_values.merge(EMEP_POD3_v5_UNECE_mean, on="Scenario_UKCEH", how="inner")
Crops_extrapolation_values = Crops_extrapolation_values[["Region", "Scenario_UKCEH", "Scenario", "Crop", "Avoided_prod_losses_weight", "POD3"]]

# Extract and separate wheat and potatoes from to be filled values
wheat_to_fill = Crops_expanded[(Crops_expanded["Crop"] == "Wheat")]
potato_to_fill = Crops_expanded[(Crops_expanded["Crop"] == "Potato")]

# Extract and separate wheat and potatoes from extrapolation values
wheat = Crops_extrapolation_values[(Crops_extrapolation_values["Crop"] == "Wheat")]
potato = Crops_extrapolation_values[(Crops_extrapolation_values["Crop"] == "Potato")]

In [34]:
def fill_avoided_prod_losses(df_to_fill, df_with_known_points):
    
    filled_df = df_to_fill.copy()
    known_sorted = df_with_known_points.sort_values("POD3").copy()

    x = known_sorted["POD3"].to_numpy(dtype=float)
    y = known_sorted["Avoided_prod_losses_weight"].to_numpy(dtype=float)

    f = interp1d(x, y, kind="linear", fill_value="extrapolate", bounds_error=False)

    mask = filled_df["Avoided_prod_losses_weight"].isna()

    filled_df.loc[mask, "Avoided_prod_losses_weight"] = f(
        filled_df.loc[mask, "POD3"].to_numpy(dtype=float)
    )

    return filled_df

wheat_filled = fill_avoided_prod_losses(wheat_to_fill, wheat)
potato_filled = fill_avoided_prod_losses(potato_to_fill, potato)

# Add all dfs into one df
crops_out_check = pd.concat([wheat_filled, potato_filled], ignore_index=True)
crops_out_check = crops_out_check[["Region", "Scenario", "Year", "Crop", "POD3", "Avoided_prod_losses_weight"]]

# Convert to common scenario names
crops_out_check = crops_out_check.replace(MAPPING_SCENARIOS)
crops_out_check = crops_out_check.replace(MAPPING_COUNTRIES)

In [35]:
# Save the df as an Excel file
crops_out_check.to_excel("Processed_Data/crops_out_check.xlsx", index=False)

# Table Conversion - Carbon

Below follows all data management specifically pertaining to the carbon uptake valuation parts of the CBA

In [36]:
# Choose the rows which ends with the relevant scenario version in use

carbon_out = avoided_carbon_losses[
    avoided_carbon_losses["Scenario"].str.endswith(DATA_VERSION.lower())
]
carbon_out = carbon_out.replace(MAPPING_SCENARIOS)

In [37]:
# Merge with carbon avoided losses
Carbon_to_fill = carbon_out.merge(EMEP_POD1_v7, on=["Scenario", "Country", "Year"], how="outer")
Carbon_to_fill = Carbon_to_fill.drop(columns=["Avoided_carbon_losses_weight", "Scenario_UKCEH", "POD1_variable", "country_code"])
Carbon_to_fill = Carbon_to_fill[Carbon_to_fill["POD1"] != 0]

# Merge with carbon avoided losses to extrapolate
EMEP_POD1_v5 = EMEP_POD1_v5.rename(columns={"Scenario": "Scenario_UKCEH"})
Carbon_to_extrapolate = Extr_carbon_values.merge(EMEP_POD1_v5, on=["Scenario_UKCEH", "Country"], how="inner")
Carbon_to_extrapolate = Carbon_to_extrapolate[["Country", "Scenario", "Item", "Avoided_carbon_losses_weight", "POD1"]]

In [38]:
df2 = Carbon_to_extrapolate[Carbon_to_extrapolate["Avoided_carbon_losses_weight"]!=0]
df2 = df2[df2["POD1"]!=0]


from scipy.interpolate import PchipInterpolator

country_functions = {}

for country, g in df2.groupby("Country"):

    g = g.sort_values("POD1")

    x = g["POD1"].values
    y = g["Avoided_carbon_losses_weight"].values

    country_functions[country] = PchipInterpolator(x, y)

In [39]:
Carbon_to_fill["Avoided_carbon_losses_weight"] = [
    max(float(country_functions[c](x)), 0)
    if c in country_functions else np.nan
    for c, x in zip(
        Carbon_to_fill["Country"],
        Carbon_to_fill["POD1"]
    )
]

# =====================================================
# FINAL OUTPUT
# =====================================================

carbon_out_check = Carbon_to_fill.copy()

carbon_out_check = carbon_out_check.replace(MAPPING_SCENARIOS)
carbon_out_check = carbon_out_check.replace(MAPPING_COUNTRIES)

In [40]:
# Save the df as an Excel file
carbon_out_check.to_excel("Processed_Data/carbon_out_check.xlsx", index=False)

# Table Conversion - Soiling

Below follows all data management specifically pertaining to the Soiling valuation parts of the CBA

### Surface area per citizen

In [41]:
# Melt the df as to only have one value column
Soiling_area_Project_est_melt = Soiling_area_Project_est[["Country", "Source", "Low", "Mid", "High"]].melt(
    id_vars=["Country", "Source"],
    var_name="Facade_level",
    value_name="Facade_area"
)

# Add all dfs into one df
Soiling_area = pd.concat([Soiling_area_ICP, Soiling_area_Project_est_melt], ignore_index=True)

# Use harmonised country names
Soiling_area = Soiling_area.replace(MAPPING_COUNTRIES)

# Merge with regions

# Insert a Region columns = Country
Soiling_area_country = Soiling_area.copy()
Soiling_area_country["Region"] = "Country"

# Expand each regional row to all countries belonging to that region
Soiling_area_regional = Soiling_area.copy()
Soiling_area_regional = Soiling_area_regional.merge(region_df, on="Country", how="inner")

# Add all dfs into one df
Soiling_area_out = pd.concat([Soiling_area_regional, Soiling_area_country], ignore_index=True)

# Reorder columns
Soiling_area_out = Soiling_area_out[["Region", "Country", "Source", "Facade_level", "Facade_area"]]

# Convert to common scenario names
Soiling_area_out = Soiling_area_out.replace(MAPPING_COUNTRIES)


In [42]:
# Save the df as an Excel file
Soiling_area_out.to_excel("Processed_Data/Soiling_area_check.xlsx", index=False)

### PM10 concentrations per m2

In [43]:
PM10_conc = EMEP_PM10[["Scenario", "Country", "Year", "PM10_surface_ug_m3"]]

# Save the df as an Excel file
PM10_conc.to_excel("Processed_Data/PM10_conc_check.xlsx", index=False)

# Table Conversion - Biodiversity & Fertilisation

Below follows all data management specifically pertaining to the Biodiversity valuation part of the CBA

## Exceedence of Emprical Critical Loads (CL)

The code below cleans the data for Exceedence of Critical Loads for Empirical Critical Loads

In [44]:
# (sheet column number for each field, per ecosystem type)
ECOSYSTEM_COLUMNS = {
    "Semi_natural": {"Area": 12, "Exceeded": 13, "AAE": 14},   # L, M, N
    "Forest":       {"Area": 6,  "Exceeded": 7,  "AAE": 8},    # F, G, H
    "Catchment":    {"Area": 9,  "Exceeded": 10, "AAE": 11},   # I, J, K
}

def read_biodiv_workbook(wb, region_df, start_row=10, end_row=57):
    """Read all scenario sheets of one workbook into one long dataframe."""
    all_frames = []

    for ws in wb.worksheets:
        if ws["A3"].value is None and ws["A5"].value is None:
            print(f"Skipping empty sheet: {ws.title}")
            continue

        scenario = str(ws["A3"].value).split(": ")[-1] or ws.title
        year = int(str(ws["A5"].value).split(": ")[-1])

        rows = range(start_row, end_row + 1)
        countries = [ws.cell(row=r, column=2).value for r in rows]   # B

        for eco_type, cols in ECOSYSTEM_COLUMNS.items():
            df = pd.DataFrame({
                "Country": countries,
                "Year": year,
                "Scenario": scenario,
                "Ecosystem_type": eco_type,
                **{field: [ws.cell(row=r, column=c).value for r in rows]
                   for field, c in cols.items()},
            })
            all_frames.append(df.dropna(subset=["Country"]))

    df_all = pd.concat(all_frames, ignore_index=True)
    df_all = df_all.replace(MAPPING_SCENARIOS).replace(MAPPING_COUNTRIES)

    # Country rows keep their own values; regional rows are expanded to member countries
    country_rows = df_all.copy()
    country_rows["Region"] = "Country"
    regional_expanded = df_all.merge(region_df, on="Country", how="inner")
    out = pd.concat([country_rows, regional_expanded], ignore_index=True)

    return out[["Region", "Country", "Year", "Scenario", "Ecosystem_type",
                "AAE", "Area", "Exceeded"]]

In [45]:
#Read and run 2040 and 2050 workbooks
out_clempn_exc_allecosys_2040 = read_biodiv_workbook(wb_biodiv_2040, region_df)
out_clempn_exc_allecosys_2050 = read_biodiv_workbook(wb_biodiv_2050, region_df)

# Merge the two dfs to minimise data outputs
out_clempn_exc_allecosys = pd.concat([out_clempn_exc_allecosys_2040, out_clempn_exc_allecosys_2050], ignore_index=True)

Skipping empty sheet: Sheet1


In [46]:
# Save the df as an Excel file
out_clempn_exc_allecosys.to_excel("Processed_Data/clempn_exc_allecosys_check.xlsx", index=False)

## Biodiversity Damage Costs

The code below cleans the data for Biodiversity Damage Costs

In [47]:
# switch the column row
df_biodiv_costs.columns = df_biodiv_costs.iloc[0]
df_biodiv_costs_header = df_biodiv_costs.iloc[1:].reset_index(drop=True)

# Clean the data and use harmonised terminology
df_biodiv_costs_header_corr = df_biodiv_costs_header.rename(columns={
    "terestrial land": "Forest",
    "meadows and pastures": "Semi_natural"
})

out_biodiv_costs = df_biodiv_costs_header_corr[["Country", "Forest", "Semi_natural"]]

out_biodiv_costs_long = out_biodiv_costs.melt(
    id_vars="Country",
    value_vars=["Forest", "Semi_natural"],
    var_name="Ecosystem_type",
    value_name="Economic_value"
)

out_biodiv_costs_clean = out_biodiv_costs_long.replace(MAPPING_COUNTRIES)

# Adding countries for which data is lacking

new_countries = ['Cyprus',
                'Russia',
                'Belarus',
                'Ukraine',
                'Bosnia-H']

source = out_biodiv_costs_clean[out_biodiv_costs_clean["Country"] == "Türkiye"]

duplicates = pd.concat(
    [source.assign(Country=c) for c in new_countries],
    ignore_index=True
)

out_biodiv_costs_check = pd.concat([out_biodiv_costs_clean, duplicates], ignore_index=True)

C:\Users\StefanÅström\AppData\Local\Temp\ipykernel_16676\281982705.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out_biodiv_costs_clean = out_biodiv_costs_long.replace(MAPPING_COUNTRIES)


In [48]:
# Save the df as an Excel file
out_biodiv_costs_check.to_excel("Processed_Data/out_biodiv_costs_check.xlsx", index=False)

# Table Conversion - Climate

Below follows all data management specifically pertaining to the Climate effect valuation in the CBA. Climate effects are derived from the radative forcing of air pollution emissions.

## Emissions

Below follows the cleaning of emissions data per country and scenario.

In [49]:
# Ignore first sheet
sheet_names = emxls.sheet_names[1:]
rows = []

# =====================================================
# Process each country sheet
# =====================================================
for sheet in sheet_names:
    # Read sheet without headers (fixed cell positions)
    df = pd.read_excel(
        emxls,
        sheet_name=sheet,
        header=None)

    # -----------------------------
    # Country (B1)
    # -----------------------------
    country = df.iloc[0, 1]

    # -----------------------------
    # Scenario (row 3, C:AD)
    # -----------------------------
    scenarios = df.iloc[2, 2:30].astype("string").tolist()

    # -----------------------------
    # Year (row 4, C:AD)
    # -----------------------------
    years = df.iloc[3, 2:30].tolist()

    # -----------------------------
    # Sector (B5:B109)
    # -----------------------------
    sectors = df.iloc[4:109, 1].astype("string")
    
    # -----------------------------
    # Pollutant (A5:A109)
    # -----------------------------
    pollutants = df.iloc[4:109, 0].astype("string")

    # -----------------------------
    # emissions (C5:AD109)
    # -----------------------------
    emissions = df.iloc[4:109, 2:30].values

    # -----------------------------
    # Clean blank seperator rows
    # -----------------------------
    mask = sectors.notna() & sectors.ne("") & pollutants.notna() & pollutants.ne("")
    sectors = sectors[mask].tolist()
    pollutants = pollutants[mask].tolist()
    emissions = emissions[mask.values, :]  # keep same rows
    
    # -----------------------------
    # Build stacked dataframe
    # -----------------------------
    for i, (sector, pollutant) in enumerate(zip(sectors, pollutants)):
        for j, (scenario, year) in enumerate(zip(scenarios, years)):
            if scenario == "" or pd.isna(year):
                continue
            rows.append({
                "Country": country,
                "Year": int(year) if pd.notna(year) else None,
                "Scenario": scenario,
                "Sector": sector,
                "Pollutant": pollutant,
                "Emissions_value": emissions[i, j]
            })

# =====================================================
# Final emissions dataframe 
# =====================================================
stacked_df = pd.DataFrame(rows)


# Ensure numeric emissions
stacked_df["Emissions_value"] = pd.to_numeric(
    stacked_df["Emissions_value"], errors="coerce"
)

# Drop incomplete rows if any
em_df = stacked_df.dropna(
    subset=["Country", "Sector", "Year", "Scenario"]
).reset_index(drop=True)

# Fill any empty value cells with 0
em_df["Emissions_value"] = em_df["Emissions_value"].fillna(0)

# Replace pollutant names
em_df.replace(to_replace="VOC", value="NMVOC", inplace=True)

# =====================================================
# Calculating PMres emissions (PM2.5 - BC) into Final emissions dataframe 
# =====================================================

# Choose the dimensions that define “same row”
KEY = ["Country", "Year", "Scenario", "Sector"] 

# Build two aligned Series (MultiIndex = KEY)
s_pm25 = (em_df.loc[em_df["Pollutant"].eq("PM2.5")]
          .set_index(KEY)["Emissions_value"])

s_bc = (em_df.loc[em_df["Pollutant"].eq("BC")]
        .set_index(KEY)["Emissions_value"])

# Subtract with alignment (missing treated as 0)
em_PMres = s_pm25.sub(s_bc, fill_value=0)

# Convert Series -> rows to append
pmres_rows = em_PMres.reset_index(name="Emissions_value")
pmres_rows["Pollutant"] = "PMres"

# Append (optionally remove old PMres first to avoid duplicates)
em_df = em_df.loc[~em_df["Pollutant"].eq("PMres")].copy()
em_df = pd.concat([em_df, pmres_rows], ignore_index=True)

# Convert units from kton per year to ton per year
em_df["Emissions_value"] = em_df["Emissions_value"]*1000

# em_df_without_CH4 = em_df[em_df["Pollutant"] != "CH4"]
em_df = em_df.replace(MAPPING_COUNTRIES)

# =====================================================
# Merge with regions
# =====================================================

# 1. Country-specific rows: keep their own values
country_rows = em_df.copy()
country_rows["Region"] = "Country"

# 3. Expand each regional row to all countries belonging to that region
regional_expanded = em_df.merge(region_df, on="Country", how="inner")

# 4. Combine both
out_em_df = pd.concat([country_rows, regional_expanded], ignore_index=True)

# 5. Keep columns in desired order
out_em_df = out_em_df[["Region", 
                       "Country", 
                       "Year", 
                       "Scenario", 
                       "Sector", 
                       "Pollutant",
                       "Emissions_value"]]

#  Rename scenarios to common names
out_em_df = out_em_df.replace(MAPPING_SCENARIOS)
out_em_df = out_em_df.replace(MAPPING_COUNTRIES)

In [50]:
# Save the df as an Excel file
out_em_df.to_excel(r"Processed_Data/emissions_check.xlsx", index=False)

## Climate change Costs

Below follows data cleaing of emissions damage costs which includes harmonising labels for pollutants and removing unnecessary columns.

In [51]:
# 1) Keep only the required columns
SLCF_c_df = df_em_costs.filter(["Discount rate", "Comparator", "Scenario year", "Unit-cost"], axis=1)

# 2) Renaming

# Columns
SLCF_c_df = SLCF_c_df.rename(columns={
    "Comparator": "Pollutant",
    "Scenario year": "Year",
    "Unit-cost": "Econ_value"
})

# Specific values
mapping_emissions = {
    "SCBC": "BC",
    "SCC": "CO2",
    "SCM": "CH4", 
    "SC-N2O": "N2O",
    "SCNH3": "NH3",
    "SCNMVOC": "NMVOC",
    "SCNOx": "NOx",
    "SCPMres": "PMres",
    "SCSO2": "SO2"}

SLCF_c_df = SLCF_c_df.replace(mapping_emissions)

In [52]:
# Save the df as an excel file
SLCF_c_df.to_excel(r"Processed_Data/SLCF_cost_check.xlsx", index=False)

## Emissions Climate Forcing Values

Below follows data cleaing of emissions climate forcing valuess which includes harmonising labels for pollutants and removing unnecessary columns. Although the source is presented from Åström & Källmark 2023, the original source is values from the IPCC Fifth Assessment Report.

In [53]:
# 2) Only keep certain columns
df_filtered = df_climate_metrics.filter(["Pollutant", "Climate_metric", "Level", "Metric_value"], axis=1)

# 3) Filter out relevant CO2 climate metric values, 
remove_metric_vals = ["GWP20-E [tonne CO2eq/tonne emission]", 
               "GWP100-E [tonne CO2eq/tonne emission]", 
               "GTP20-E [tonne CO2eq/tonne emission]", 
               "GTP100-E [tonne CO2eq/tonne emission]"]

df_filtered = df_filtered[
    df_filtered["Climate_metric"].isin(remove_metric_vals) == False]

# 4) Filter uncessesary pollutants, 
remove_poll_vals = ["PMOC", "CO2"]

df_filtered = df_filtered[
    df_filtered["Pollutant"].isin(remove_poll_vals) == False]

# Rename specific values in one column (exact matches)
mapping_SLCF = {
    "PMBC": "BC",
    "PMRES": "PMres",
    "High (95th perc.)": "High",
    "Low (5th perc.)": "Low",
    "PMBC": "BC",
    "GWP20-E [tonne CH4eq/tonne emission]": "GWP20-E",
    "GWP100-E [tonne CH4eq/tonne emission]": "GWP100-E",
    "GTP20-E [tonne CH4eq/tonne emission]": "GTP20-E",
    "GTP100-E [tonne CH4eq/tonne emission]": "GTP100-E"
}
SLCF_m_df= df_filtered.replace(mapping_SLCF)

In [54]:
# Save the df as an excel file
SLCF_m_df.to_excel(r"Processed_Data/Metric_check.xlsx", index=False)

# Table Conversion - Health

Below follows all data management specifically pertaining to the Health valuation part of the CBA

## PM2.5 Populated weighted annual average concentrations

Here follows data cleaning of PM2.5 population-weighted annual average concentrations data where non-relevant columns are filtered out.

In [55]:
def read_pm25_workbook(wb, start_row=10, end_row=61):
    """Read all scenario sheets of one PM2.5 workbook into one long dataframe."""
    all_frames = []

    for ws in wb.worksheets:
        scenario = str(ws["A3"].value).split(": ")[-1] or ws.title
        year = int(str(ws["A5"].value).split(": ")[-1])

        rows = range(start_row, end_row + 1)
        df = pd.DataFrame({
            "Country": [ws.cell(row=r, column=2).value for r in rows],   # B
            "Year": year,
            "Scenario": scenario,
            "Mean PM2.5 concentration_value": [ws.cell(row=r, column=4).value for r in rows],  # D
            "YOLL_value": [ws.cell(row=r, column=5).value for r in rows],                       # E
            "Premat_deaths": [ws.cell(row=r, column=8).value for r in rows],                    # H
        })
        all_frames.append(df.dropna(subset=["Country"]))

    out = pd.concat(all_frames, ignore_index=True)
    return out.replace(MAPPING_COUNTRIES).replace(MAPPING_SCENARIOS)

In [56]:
PM25_df_2040 = read_pm25_workbook(wb_pm_conc_2040)
PM25_df_2050 = read_pm25_workbook(wb_pm_conc_2050)

# Merge the two dfs to minimise data outputs
PM25_df = pd.concat([PM25_df_2040, PM25_df_2050], ignore_index=True)


In [57]:
# Save the df as an excel file
PM25_df.to_excel(r"Processed_Data/PM25_check.xlsx", index=False)

## Exposure Respone Function (ERF)

Cleaning of data and harmonising mortality and mortality nomenclature.

In [58]:
# Select only the relevant columns from the original ERF dataframe
df_filtered = df_H_ERF.filter(['Pollutant','Core?','Health_outcome', 'Exposure window', 'Show age', 'ERF_L', 'ERF_C', 'ERF_H'], axis=1)

# Remove rows corresponding to short-term exposure, keeping only the other exposure types
df_filtered = df_filtered[df_filtered["Exposure window"] != "Short-term"]

# Drop the exposure window column since it is no longer needed after filtering
df_filtered = df_filtered.drop(columns=["Exposure window"])

# Rename columns to simpler and more user-friendly names
df_filtered.rename(columns={"Core?": "Core", "Show age": "Age group", "ERF_L": "Low", "ERF_C": "Mid", "ERF_H": "High"}, inplace=True)

# Reshape the dataframe from wide to long format:
# the ERF level columns (Low, Mid, High) are stacked into two columns:
# one for the effect level label and one for the corresponding value
df_long = df_filtered.melt(
    
    id_vars=["Pollutant", "Core", "Health_outcome", 'Age group'],
    var_name="Effect_level",
    value_name="Effect_value"
)

# Filter the long-format dataframe to keep only PM2.5-related rows
df_long = df_long[
    df_long["Pollutant"].isin(["Fine particles (PM2.5)"]) == True]

# Define a mapping to simplify and standardize pollutant and health outcome names
mapping = {
    "Fine particles (PM2.5)": "PM2.5",
    "Natural mortality": "All-cause mortality",
    "Incidence of ALRI in children": "ALRI in children",
    "Incidence of asthma in adults": "Asthma in adults",
    "Incidence of asthma in children": "Asthma in children",
    "Incidence of COPD": "COPD",
    "Incidence of Dementia": "Dementia",
    "Incidence of Diabetes": "Diabetes",
    "Incidence of IHD events": "IHD events",
    "Incidence of Lung Cancer": "Lung cancer",
    "Incidence of Stroke": "Stroke"
}

# Apply the text replacements across the dataframe
df_long = df_long.replace(mapping)

# Create a new column classifying whether the health outcome is all-cause mortality
# or a specific morbidity/mortality cause
df_long["Mor_cause"] = np.where(
    df_long["Health_outcome"] == "All-cause mortality",
    "All",
    "Specific"
)

# Replace country and scenario names
df_long = df_long.replace(MAPPING_COUNTRIES)

In [59]:
# Save the df as an excel file
df_long.to_excel(r"Processed_Data/Health_ERF_check.xlsx", index=False)

## Morbidity Costs

The code below filters and merges morbidity costs from different sources, harmonises naming with the other formats and interpolates values for any missing countries.

In [60]:
# Select relevant cost columns from both datasets
df_filtered = df_morb_costs.filter(['Country/region','Damage type','Healthcare', 'Productivity loss', 'Disutility'], axis=1)
df_filtered_nordics = df_morb_nordics_costs.filter(['Country/region','Damage type','Healthcare', 'Productivity loss', 'Disutility'], axis=1)

# Combine the two datasets into one unified dataframe
df_all = pd.concat([df_filtered, df_filtered_nordics], ignore_index=True)

# Reshape the dataframe from wide to long format:
# cost categories (Healthcare, Productivity loss, Disutility) are stacked into one column
df_long = df_all.melt(
    id_vars=['Country/region','Damage type'],
    var_name="Cost_type",
    value_name="Mor_value"
)

# Rename columns to standardized names for consistency with other datasets
df_long.rename(columns={"Country/region": "Country", "Damage type": "Health_outcome"}, inplace=True)
df_long = df_long.replace(MAPPING_COUNTRIES)

# Add rows for Kosovo by duplicating Serbia's values (assumption: Serbia values apply to Kosovo)
kosovo = (df_long.loc[df_long["Country"] == "Serbia"]).copy()
kosovo.replace(to_replace="Serbia", value="Kosovo", inplace=True)
df_long = pd.concat([df_long, kosovo], ignore_index=True)


# Apply the text replacements across the dataframe
df_long = df_long.replace(mapping)

# 1. Create country-specific rows and label them explicitly as "Country"
country_rows = df_long.copy()
country_rows["Region"] = "Country"

# 2. Create regional rows by treating the Country column as a Region column
regional_rows = df_long.rename(columns={"Country": "Region"}).copy()

# 3. Expand each regional row to all countries belonging to that region
# by merging with a region-to-country mapping table
regional_expanded = regional_rows.merge(region_df, on="Region", how="inner")

# 4. Combine country-specific rows with expanded regional rows
out_morb_costs = pd.concat([country_rows, regional_expanded], ignore_index=True)

# 5. Reorder columns to the desired output structure
out_morb_costs = out_morb_costs[["Region", "Country", "Health_outcome", "Cost_type", "Mor_value"]]

# 6. Remove rows where the Country column actually contains region names (cleanup step)
region_names = region_df["Region"].unique()
out_morb_costs = out_morb_costs[~out_morb_costs["Country"].isin(region_names)]

# Replace country and scenario names
out_morb_costs = out_morb_costs.replace(MAPPING_COUNTRIES)

In [61]:
# Save the df as an excel file
out_morb_costs.to_excel(r"Processed_Data/Morb_costs_check.xlsx", index=False)

## Mortality values

The code below filters and merges mortality costs from different sources, harmonises naming with the other formats and interpolates values for any missing countries.

In [62]:
# Reshape the dataframe from wide to long format:
# cost categories (VOLY and VSL) are stacked into one column
df_long = df_mort_costs_oecd.melt(
    id_vars=['Country/region'],
    var_name="Cost_type",
    value_name="Mor_value"
)

# Rename columns to standardized names for consistency with other datasets
df_long.rename(columns={"Country/region": "Country"}, inplace=True)
df_long = df_long.replace(MAPPING_COUNTRIES)

# Add rows for Kosovo by duplicating Serbia's econmic values (assumption: Serbia values apply to Kosovo)
kosovo = (df_long.loc[df_long["Country"] == "Serbia"]).copy()
kosovo.replace(to_replace="Serbia", value="Kosovo", inplace=True)
df_long = pd.concat([df_long, kosovo], ignore_index=True)


# Apply the text replacements across the dataframe
df_long = df_long.replace(mapping)

# 1. Country-specific rows: keep their own values
country_rows = df_long.copy()
country_rows["Region"] = "Country"

# 2. Regional rows: rows where Country contains a region label instead of an actual country
regional_rows = df_long.rename(columns={"Country": "Region"}).copy()

# 3. Expand each regional row to all countries belonging to that region
regional_expanded = regional_rows.merge(region_df, on="Region", how="inner")

# 4. Combine both
out_mort_costs = pd.concat([country_rows, regional_expanded], ignore_index=True)

# 5. Keep columns in desired order
out_mort_costs = out_mort_costs[["Region", "Country", "Cost_type", "Mor_value"]]

# 6. Drop rows with regions in Country-column
region_names = region_df["Region"].unique()
out_mort_costs = out_mort_costs[~out_mort_costs["Country"].isin(region_names)]

# Replace country and scenario names
out_mort_costs = out_mort_costs.replace(MAPPING_COUNTRIES)

In [64]:
# Save the df as an excel file
out_mort_costs.to_excel(r"Processed_Data/Mort_costs_check.xlsx", index=False)

## Mortality Incidents Rate (Inc_R) per Cohort

Here follows the determination of mortality incidents rate per cohort and seperated between all-cause and specific-cause mortality.

### All-cause Mortality

In [65]:
# Select columns that are integers (age columns)
age_cols_pop = [c for c in df_mort_all_eff.columns if isinstance(c, int)]

age_cols_0_99 = age_cols_pop[:-1]

# Create age groups: 0, 1, 2-4, then 5-year intervals
groups = {
    0: [0],
    1: [1],
    "2-4": [2, 3, 4]
}

# Continue with normal 5-year groups from age 5
groups.update({
    f"{i}-{i+4}": list(range(i, i+5))
    for i in range(5, 100, 5)
})

# Sum columns within each group
df_mort_all_5yr = pd.concat(
    [df_mort_all_eff.drop(columns=age_cols_pop),
     pd.DataFrame({name: df_mort_all_eff[cols].sum(axis=1) for name, cols in groups.items()})],
    axis=1
)

df_mort_all_5yr.rename(columns={"95-99": "95+"}, inplace=True)

df_mort_all_5yr["Health_outcome"] = "All-cause mortality"

### Cause-Specific Mortality

In [66]:
df_mort_spec_eff.rename(columns={"Death cause": "Health_outcome"}, inplace=True)

# Melting the df as to only have one value column
df_mort_spec_long = df_mort_spec_eff.melt(
    id_vars=['Country', 'Year', 'Health_outcome'],
    var_name="5yr cohort",
    value_name="Mort_value"
)

# Identify the min, mean, and max values for each set of "Country", "Health_outcome", "Age group"
df_mort_spec_summary = (
    df_mort_spec_long
    .groupby(["Country", "Health_outcome", "5yr cohort"], as_index=False)
    ["Mort_value"]
    .agg(
        Low="min",
        Mid="mean",
        High="max"
    ))

# Melting the df as to only have one value column
df_mort_spec_summary_long = df_mort_spec_summary.melt(
    id_vars=['Country', 'Health_outcome', "5yr cohort"],
    var_name="Mort_level",
    value_name="Mort_value"
)

# Rename the health outcomes to match the VALESOR terminology
mapping_mort = {
    "IHD = Ischemic heart disease events": "IHD mortality",
    "CVD = cardiovascular diseases": "Cardiovascular mortality",
    "Respiratory diseases": "Respiratory mortality",
    "COPD = Chronic obstructive pulmonary disease": "COPD mortality",
    "ALRI = Acute lower respiratory infections": "ALRI mortality",
    "Lung cancer": "Lung cancer mortality", 
    "Stroke": "Stroke mortality"
}


df_mort_spec_summary_long = df_mort_spec_summary_long.replace(mapping_mort)

### Merge and Manage Together

In [67]:
# Reshape mortality data from wide to long format
df_mort_all_long = df_mort_all_5yr.melt(
    id_vars=["Country", "Year", "Health_outcome"],
    var_name="5yr cohort",
    value_name="Mort_value"
)

# Split into cohort groups based on how values should be adjusted
df_mort_cohort_unchanged = df_mort_all_long[df_mort_all_long["5yr cohort"].isin([0, 1])]

df_mort_cohort_2_4 = df_mort_all_long[df_mort_all_long["5yr cohort"].isin(["2-4"])]
df_mort_cohort_2_4["Mort_value"] = df_mort_cohort_2_4["Mort_value"] / 2

df_mort_cohort_5yr = df_mort_all_long[~df_mort_all_long["5yr cohort"].isin([0, 1, "2-4"])]
df_mort_cohort_5yr["Mort_value"] = df_mort_cohort_5yr["Mort_value"] / 5

# Combine all adjusted cohort groups into one dataframe
df_mort_all_adjusted = pd.concat(
    [df_mort_cohort_unchanged, df_mort_cohort_2_4, df_mort_cohort_5yr],
    ignore_index=True
)

# Apply overall adjustment factor
adjust_factor = 0.95
df_mort_all_adjusted["Mort_value"] = df_mort_all_adjusted["Mort_value"] * adjust_factor

# Combine all-cause mortality data with specific mortality summary data
df_mort = pd.concat([df_mort_spec_summary_long, df_mort_all_adjusted], ignore_index=True)

# Fill missing values with defaults
df_mort["Mort_level"] = df_mort["Mort_level"].fillna("Mid")
df_mort["Year"] = df_mort["Year"].fillna(2040)

# Reorder columns into the desired output format
df_mort = df_mort[["Country", "Year", "Health_outcome", "5yr cohort", "Mort_level", "Mort_value"]]

# Replace country and scenario names
df_mort = df_mort.replace(MAPPING_COUNTRIES)

C:\Users\StefanÅström\AppData\Local\Temp\ipykernel_16676\3241832886.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mort_cohort_2_4["Mort_value"] = df_mort_cohort_2_4["Mort_value"] / 2
C:\Users\StefanÅström\AppData\Local\Temp\ipykernel_16676\3241832886.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mort_cohort_5yr["Mort_value"] = df_mort_cohort_5yr["Mort_value"] / 5


In [68]:
# Save the df as an excel file
df_mort.to_excel(r"Processed_Data/Mort_incidents.xlsx", index=False)

## Morbidity Incidents Rate (Inc_R) per Cohort

Here follows the determination of morbidity incidents rate per cohort. As no future values are available, the method chooses the highest, mean, and lowest values from the available historical years and sets them as High, Mid, Low.

In [69]:
df_morb_eff.rename(columns={"Desease": "Health_outcome"}, inplace=True)

# Melting the df as to only have one value column
df_morb_long = df_morb_eff.melt(
    id_vars=['Country', 'Year', 'Health_outcome'],
    var_name="5yr cohort",
    value_name="Morb_value"
)

# Identify Asthma cases pertaining to children versus adults.
mask_asthma_child = (
    (df_morb_long["Health_outcome"] == "Asthma") &
    (df_morb_long["5yr cohort"].isin([0, 1, "2-4", "5-9", "10-14", "15-19"]))
)
mask_asthma_adult = (
    (df_morb_long["Health_outcome"] == "Asthma") &
    (df_morb_long["5yr cohort"].isin(['20-24', '25-29', '30-34',
       '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69',
       '70-74', '75-79', '80-84', '85-89', '90-94', '95+']))
)

# Rename accordingly
df_morb_long.loc[mask_asthma_child, "Health_outcome"] = "Asthma in children"
df_morb_long.loc[mask_asthma_adult, "Health_outcome"] = "Asthma in adults"

# Rename the health outcomes to match the VALESOR terminology
mapping_morb = {
    "Incidence of Diabetes": "Diabetes",
}

df_morb_long = df_morb_long.replace(mapping_morb)

# Identify the min, mean, and max values for each set of "Country", "Health_outcome", "Age group"
df_morb_summary = (
    df_morb_long
    .groupby(["Country", "Health_outcome", "5yr cohort"], as_index=False)
    ["Morb_value"]
    .agg(
        Low="min",
        Mid="mean",
        High="max"
    ))

# Melting the df as to only have one value column
df_morb_summary_long = df_morb_summary.melt(
    id_vars=['Country', 'Health_outcome', "5yr cohort"],
    var_name="Morb_level",
    value_name="Morb_value"
)

# Place columns in desired order
df_morb_summary_long = df_morb_summary_long[["Country", "Health_outcome", "5yr cohort", "Morb_level", "Morb_value"]]

# Replace country and scenario names
df_morb_summary_long = df_morb_summary_long.replace(MAPPING_COUNTRIES)


In [70]:
# Save the df as an excel file
df_morb_summary_long.to_excel(r"Processed_Data/Morb_incidents.xlsx", index=False)